# Fotoğraf dokusundan kütle yüzdelerine karşılık gelen çapları tahmin etme
Aynı eğitim verisi ve aynı Ridge ayarlarıyla iki hedef biçimini karşılaştırıyoruz.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.preprocessing import StandardScaler

root = Path("/kaggle/input") if Path("/kaggle/input").exists() else next(p for p in [Path("work/soil-data"), Path("../work/soil-data")] if p.exists())
data = next(root.rglob("Training_labels_updated.csv")).parent
labels = pd.read_csv(data / "Training_labels_updated.csv").set_index("sample_id")
cameras = pd.read_csv(data / "ppm_updated.csv")
photos = sorted(data.joinpath("Training-All_Photos_updated").rglob("*.jpg"))
print(f"{len(labels)} numune, {len(photos)} eğitim fotoğrafı")

24 numune, 127 eğitim fotoğrafı


## 1. Fotoğrafı bir avuç sayıyla anlatmak
Bütün fotoğrafların merkezinden yaklaşık 100 × 100 mm alan alıyoruz. Kamera tablosundaki piksel/mm değerini gerçek dosyanın çözünürlüğüne uyarlıyoruz. Bu kalibrasyon, çekim uzaklığının kamera tablosunda varsayılana yakın olduğunu kabul eder.

Bu alanı griye çevirip 400 × 400 piksele getiriyoruz. Yatay ve dikey yönde, birbirinden 0,25 mm ile 8 mm arasında uzakta olan noktaların ne kadar farklı aydınlıkta göründüğünü ölçüyoruz. Böylece her fotoğrafta aynı altı fiziksel uzaklığı karşılaştırıyoruz. Her numunenin fotoğraflarını, her ölçünün ortanca değerini alarak birleştiriyoruz.

In [2]:
def physical_patch(path, camera):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        w, h = image.size
        pixels_per_mm = camera.ppm * max(w, h) / max(camera.width, camera.height)
        side = round(100 * pixels_per_mm)
        assert 0 < side <= min(w, h)
        x, y = (w - side) // 2, (h - side) // 2
        return image.crop((x, y, x + side, y + side)).convert("RGB").resize((400, 400), Image.Resampling.LANCZOS)

def texture(patch):
    gray = np.asarray(patch.convert("L"), dtype=float)
    low, high = np.percentile(gray, [5, 95])
    gray = np.clip((gray - low) / max(high - low, 1.0), 0, 1)
    values = []
    for offset in [1, 2, 4, 8, 16, 32]:
        horizontal = np.abs(gray[:, offset:] - gray[:, :-offset]).mean()
        vertical = np.abs(gray[offset:, :] - gray[:-offset, :]).mean()
        values.append((horizontal + vertical) / 2)
    return values

camera_by_name = {c.phone.replace(" ", "_").lower(): c for c in cameras.itertuples()}
rows = []
examples = {}
for path in photos:
    phone, sample, number = path.stem.rsplit("_", 2)
    patch = physical_patch(path, camera_by_name[phone.lower()])
    rows.append([sample, *texture(patch)])
    examples.setdefault(sample, patch)
features = pd.DataFrame(rows, columns=["sample_id", "0.25", "0.5", "1", "2", "4", "8"])
features = features.groupby("sample_id").median()
assert set(features.index) == set(labels.index)
features = features.loc[labels.index]

## 2. Gerçek cevabı saklayıp tahmin yapmak
Her turda yalnızca geride kalan 23 numuneden ölçekleme değerlerini öğreniyoruz. Altı doku ölçüsüne göre uzaklık hesaplayıp en yakın üç komşuyu seçiyoruz. Komşuların ölçümlerinin ortalaması tahminimiz oluyor. Karşılaştırma için aynı 23 numunenin tamamının ortalamasını da hesaplıyoruz.

In [3]:
sizes = labels.columns.astype(float).to_numpy()
widths = np.diff(np.log10(sizes))

def emd(observed, predicted):
    return np.abs(observed[:-1] - predicted[:-1]) @ widths

scores, neighbors, predictions, averages = [], {}, {}, {}
for sample in labels.index:
    train = features.drop(index=sample)
    scaler = StandardScaler().fit(train)
    distance = np.linalg.norm(scaler.transform(train) - scaler.transform(features.loc[[sample]]), axis=1)
    nearest = train.index[np.argsort(distance, kind="stable")[:3]].tolist()
    predicted = labels.loc[nearest].mean().to_numpy()
    baseline = labels.loc[train.index].mean().to_numpy()
    observed = labels.loc[sample].to_numpy()
    assert sample not in nearest and len(set(nearest)) == 3
    assert np.all(np.diff(predicted) >= -1e-10) and np.isclose(predicted[-1], 100)
    scores.append((sample, emd(observed, baseline), emd(observed, predicted)))
    neighbors[sample], predictions[sample], averages[sample] = nearest, predicted, baseline
scores = pd.DataFrame(scores, columns=["Numune", "Ortalama", "Üç komşu"]).set_index("Numune")
print(scores.mean().round(2).to_string())
print(f"Üç komşunun daha az hata yaptığı numune sayısı: {(scores['Üç komşu'] < scores['Ortalama']).sum()}/{len(scores)}")

Ortalama    89.89
Üç komşu    44.74
Üç komşunun daha az hata yaptığı numune sayısı: 18/24


## 3. Ridge ile tahmin

Ridge her tane çapı için altı doku ölçüsünden bir yüzde tahmini öğrenir. Büyük katsayıları cezalandırma gücünü belirleyen alfa değerini, her turda yalnızca eğitim için ayrılmış 23 numune içinde seçiyoruz. Dört adayımız 0,1, 1, 10 ve 100. Her adayın başarısını içeride bir numuneyi sırayla ayırıp geçerli tahmin eğrisinin EMD hatasını ölçerek karşılaştırıyoruz.

Çıktıyı 0 ile 100 arasında tutuyoruz. Azalan yüzdeler çıkarsa izotonik regresyonla eğriyi azalan kısımları düzeltecek biçimde düzenliyoruz. Son değer her zaman 100.

In [4]:
from sklearn.isotonic import isotonic_regression
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

def ridge_curve(x, y, query, alpha):
    model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    model.fit(x, y[:, :-1])
    raw = model.predict(query)[0]
    valid = isotonic_regression(raw, increasing=True, y_min=0, y_max=100)
    return np.append(valid, 100.0)

x = features.to_numpy()
y = labels.to_numpy()
ids = np.arange(len(x))
alphas = [0.1, 1, 10, 100]
curves = []
chosen = []
for holdout in ids:
    train_ids = ids[ids != holdout]
    inner_scores = []
    for alpha in alphas:
        inner_errors = []
        for check in train_ids:
            fit_ids = train_ids[train_ids != check]
            curve = ridge_curve(x[fit_ids], y[fit_ids], x[[check]], alpha)
            inner_errors.append(emd(y[check], curve))
        inner_scores.append(np.mean(inner_errors))
    alpha = alphas[np.argmin(inner_scores)]
    chosen.append(alpha)
    curves.append(ridge_curve(x[train_ids], y[train_ids], x[[holdout]], alpha))

ridge = pd.DataFrame(curves, index=labels.index, columns=labels.columns)
assert ridge.shape == (24, 11) and np.isfinite(ridge).all().all()
assert np.all(np.diff(ridge, axis=1) >= -1e-10) and ridge["200"].eq(100).all()
assert ridge.ge(0).all().all() and ridge.le(100).all().all()
scores["Ridge"] = np.abs((ridge - labels).to_numpy()[:, :-1]) @ widths
assert round(scores["Üç komşu"].mean(), 2) == 44.74
print(scores.mean().round(2).to_string())
print("Ridge daha düşük hata:", (scores["Ridge"] < scores["Üç komşu"]).sum(), "/", len(scores))
print("H038 · 2 mm altı (%)", labels.loc["H038", "2"], predictions["H038"][list(labels.columns).index("2")], ridge.loc["H038", "2"])
print(scores.round(2).to_string())
print("Seçilen alfa ve tur sayısı:", pd.Series(chosen).value_counts().sort_index().to_dict())

Ortalama    89.89
Üç komşu    44.74
Ridge       48.01
Ridge daha düşük hata: 12 / 24
H038 · 2 mm altı (%) 47.2345 85.14643333333333 81.97379507066918
        Ortalama  Üç komşu   Ridge
Numune                            
F827      142.56     63.98   53.22
G190       74.51     24.85   12.12
H030       74.21     44.43   36.38
H037       17.53     60.27   56.15
H038       57.24     93.28  102.04
H126      102.06     14.28   20.96
H181       85.69     20.89   27.87
H183       92.48     28.41   21.86
H366       87.50      8.11   75.81
H367       93.16      7.67   12.76
H368      106.93     14.50   19.44
H371       85.66     14.03   40.35
H372       87.66      7.27   27.50
H374      100.80    128.87   94.17
H405      135.62    133.53   97.97
H493       93.35     27.70   21.82
H516       83.00     90.35   79.75
H549      117.29     13.97   38.72
H615      102.08     11.57   31.71
H616       80.52     85.30   74.30
H617       68.15     71.29   43.41
H637       60.87     40.53   30.11
H666      

## Bir toprağın kütlesini yüz eşit payla gösterme

Yüzdelik noktalarını deneyi çalıştırmadan önce %0,5, %1,5, …, %99,5 olarak seçtik. Laboratuvar yüzdeleri arasında logaritmik çap ekseninde doğrusal ara değer alıyoruz. En küçük elekten geçen payın çapını 0,002 mm sınırında tutuyoruz. Modelin çıktısı her pay için bir logaritmik çap. Tahmini tersine çevirmek için her elekten kaç payın geçtiğini sayıyoruz. Böylece yüzdeler birer puanlık adımlarla değişebilir. Aşağıdaki dönüşüm kontrolü, hiç model kullanmadan bu yaklaşık gösterimin tek başına yarattığı hatayı ölçer.

In [5]:
log_sizes = np.log10(sizes)
levels = np.arange(100) + 0.5
assert np.all(np.diff(y, axis=1) >= 0) and np.all(y[:, -1] == 100)

def to_log_diameters(curve):
    upper = np.searchsorted(curve, levels, side="left")
    lower = np.maximum(upper - 1, 0)
    output = log_sizes[upper].copy()
    between = upper > 0
    hi, lo = upper[between], lower[between]
    fraction = (levels[between] - curve[lo]) / (curve[hi] - curve[lo])
    output[between] = log_sizes[lo] + fraction * (log_sizes[hi] - log_sizes[lo])
    return output

def to_percentages(raw):
    ordered = isotonic_regression(raw, increasing=True, y_min=log_sizes[0], y_max=log_sizes[-1])
    return np.searchsorted(ordered, log_sizes, side="right") * (100 / len(levels))

diameters = np.stack([to_log_diameters(row) for row in y])
restored = np.stack([to_percentages(row) for row in diameters])
assert np.abs(restored - y).max() <= 0.500001
print("Model kullanmadan dönüşümün ortalama EMD’si:", round(np.mean([emd(a, b) for a, b in zip(y, restored)]), 3))

Model kullanmadan dönüşümün ortalama EMD’si: 0.975


## Yüzde tahmini yapan eski Ridge ile çap tahmini yapan yeni Ridge

Üstte çalıştırılan yüzde Ridge ile aynı dört alfa adayını kullanıyoruz. Dış turda saklanan toprağın fotoğraflarını ve hedeflerini model eğitiminden ve alfa seçiminden çıkarıyoruz. İç turlarda çap tahminini tekrar yüzdelere çevirip yarışmanın yerel EMD hesabını uyguluyoruz. İki yöntemin de hedefi sonunda aynı 11 noktadaki hatayı azaltmak.

In [6]:
def diameter_curve(train_idx, sample_idx, alpha):
    estimator = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    estimator.fit(x[train_idx], diameters[train_idx])
    return to_percentages(estimator.predict(x[[sample_idx]])[0])

diameter_curves, diameter_alphas = [], []
for holdout in ids:
    pool = ids[ids != holdout]
    losses = []
    for alpha in alphas:
        errors = [emd(y[check], diameter_curve(pool[pool != check], check, alpha)) for check in pool]
        losses.append(np.mean(errors))
    best_alpha = alphas[np.argmin(losses)]
    diameter_alphas.append(best_alpha)
    diameter_curves.append(diameter_curve(pool, holdout, best_alpha))

diameter_curves = pd.DataFrame(diameter_curves, index=labels.index, columns=labels.columns)
assert diameter_curves.shape == (24, 11) and np.isfinite(diameter_curves).all().all()
assert np.all(np.diff(diameter_curves, axis=1) >= 0) and diameter_curves["200"].eq(100).all()
assert diameter_curves.ge(0).all().all() and diameter_curves.le(100).all().all()
assert round(scores["Ridge"].mean(), 2) == 48.01
comparison = scores[["Ridge", "Üç komşu"]].copy()
comparison["Çap Ridge"] = np.abs((diameter_curves - labels).to_numpy()[:, :-1]) @ widths
comparison["Kazanç"] = comparison["Ridge"] - comparison["Çap Ridge"]
print(comparison.round(2).to_string())
print("Ortalamalar:", comparison.mean().round(3).to_dict())
print("Önceki Ridge'e göre iyileşen / eşit / kötüleşen:", (comparison.Kazanç > 1e-8).sum(), (comparison.Kazanç.abs() <= 1e-8).sum(), (comparison.Kazanç < -1e-8).sum())
print("Seçilen alfa ve tur sayısı:", pd.Series(diameter_alphas).value_counts().sort_index().to_dict())
print("H038'de 2 mm altı kütle yüzdesi (ölçüm / eski Ridge / çap Ridge):", labels.at["H038", "2"], round(ridge.at["H038", "2"], 2), diameter_curves.at["H038", "2"])

         Ridge  Üç komşu  Çap Ridge  Kazanç
Numune                                     
F827     53.22     63.98      61.98   -8.76
G190     12.12     24.85      17.59   -5.47
H030     36.38     44.43      13.97   22.41
H037     56.15     60.27      57.65   -1.50
H038    102.04     93.28     102.32   -0.28
H126     20.96     14.28      30.82   -9.86
H181     27.87     20.89      21.33    6.54
H183     21.86     28.41      22.89   -1.03
H366     75.81      8.11      66.57    9.24
H367     12.76      7.67      12.80   -0.04
H368     19.44     14.50      18.71    0.73
H371     40.35     14.03      46.25   -5.90
H372     27.50      7.27      28.34   -0.84
H374     94.17    128.87      92.74    1.42
H405     97.97    133.53      98.32   -0.35
H493     21.82     27.70      34.94  -13.12
H516     79.75     90.35      76.06    3.69
H549     38.72     13.97      41.29   -2.57
H615     31.71     11.57      37.89   -6.18
H616     74.30     85.30      61.19   13.12
H617     43.41     71.29      26